## Chroma CRUD Operations

In [2]:
import os
import shutil
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

## Set up paths and the vector store

In [3]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('e:/Desktop/RAG/chromadb')

In [4]:
## load environment variables from the local .env file
load_dotenv()

True

In [5]:
# use a fixed collection name and persistence path.

collection_name = "demo"
persist_dir = project_root / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persis directory: {persist_dir}")

Collection name: demo
Persis directory: e:\Desktop\RAG\chromadb\chroma_langchain_db


In [6]:
# start fresh so CRUD flow produces the same result each time
if persist_dir.exists():
    shutil.rmtree(persist_dir)
    print("Removed the old Chroma directory")
else:
    print("No previous Chroma directory was found")

No previous Chroma directory was found


In [7]:
## Create the embedding model and connect it to a persistent chroma store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_dir)
)

print("Vector store is ready")

Vector store is ready


## Add small helper functions

In [14]:
def preview_text(text,  limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."

def print_documents(title, docs):
    """Print Document objects in a beginner-friendly format"""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. id={doc.id}")
        print(f"  topic={doc.metadata.get('topic')} | doc_number={doc.metadata.get('doc_number')}")
        print(f"  content={doc.page_content}")
    print()

## Create and Insert Example Documents

In [15]:
# Keep the raw sample data separate from the Document objects so it is easier to read.
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

print(f"Prepared {len(document_examples)} document examples.")

Prepared 10 document examples.


In [16]:
# convert the sample data into langchain Document objects
documents = [
    Document(
        id=str(uuid4()),
        page_content=item['text'],
        metadata={"topic": item['topic'], "doc_number": item['doc_number']},
    )
    for item in document_examples
]

In [18]:
print_documents("dummy docs",documents)

dummy docs
1. id=68a3c254-572e-4b13-841e-5a3b13d010ae
  topic=AI | doc_number=1
  content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
2. id=85c1ed09-19b7-47a0-a756-71d9d44d56de
  topic=AI | doc_number=2
  content=AI systems can analyze patterns in data to support predictions and automation.
3. id=fd837776-5bae-47f9-8ffc-7d951da17fe9
  topic=AI | doc_number=3
  content=Responsible AI development includes fairness, transparency, and safety checks.
4. id=20031b4b-6824-4527-8e99-66b084cd48fc
  topic=RAG | doc_number=4
  content=RAG combines retrieval with generation so the model can answer using external knowledge.
5. id=1e67aaf7-2a83-4c7b-bb84-9ca3e47263a3
  topic=RAG | doc_number=5
  content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
6. id=435e964b-65c4-475d-88fb-f0de1c87c08a
  topic=RAG | doc_number=6
  content=Vector stores are important in RAG because they make semantic search over embe

In [19]:
## Insert the documents into Chroma.
document_ids = vector_store.add_documents(documents)

print("Inserted document ids:")
for doc_id in document_ids:
    print(doc_id)
    
print(f"\nTotal inserted documents: {len(document_ids)}")

Inserted document ids:
68a3c254-572e-4b13-841e-5a3b13d010ae
85c1ed09-19b7-47a0-a756-71d9d44d56de
fd837776-5bae-47f9-8ffc-7d951da17fe9
20031b4b-6824-4527-8e99-66b084cd48fc
1e67aaf7-2a83-4c7b-bb84-9ca3e47263a3
435e964b-65c4-475d-88fb-f0de1c87c08a
fb541605-0f78-4f91-8418-25027d9f081f
9722a2a8-ca98-454e-b504-2d02b4ed3952
1713080c-7c8b-4f40-bd4c-49b24fab3ed9
cebc3992-fb84-4714-83d1-23e0dbf11189

Total inserted documents: 10


## Read and Stored Data Back

In [21]:
raw_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
raw_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [22]:
print(raw_records)

{'ids': ['68a3c254-572e-4b13-841e-5a3b13d010ae', '85c1ed09-19b7-47a0-a756-71d9d44d56de', 'fd837776-5bae-47f9-8ffc-7d951da17fe9', '20031b4b-6824-4527-8e99-66b084cd48fc', '1e67aaf7-2a83-4c7b-bb84-9ca3e47263a3', '435e964b-65c4-475d-88fb-f0de1c87c08a', 'fb541605-0f78-4f91-8418-25027d9f081f', '9722a2a8-ca98-454e-b504-2d02b4ed3952', '1713080c-7c8b-4f40-bd4c-49b24fab3ed9', 'cebc3992-fb84-4714-83d1-23e0dbf11189'], 'embeddings': array([[ 0.00486374,  0.02099609,  0.01657104, ...,  0.00072002,
        -0.0164032 ,  0.0279541 ],
       [-0.0144577 , -0.00542068,  0.03039551, ..., -0.02857971,
        -0.00191212,  0.04272461],
       [ 0.02268982,  0.01535797,  0.04550171, ...,  0.02297974,
         0.00804138, -0.01585388],
       ...,
       [ 0.00804138,  0.02316284,  0.02172852, ..., -0.02069092,
        -0.01786804,  0.01335144],
       [ 0.00883484,  0.06268311,  0.09967041, ..., -0.01579285,
        -0.0137558 ,  0.03677368],
       [ 0.01806641,  0.08551025,  0.04736328, ..., -0.02371216,

In [23]:
print(f"Total records in collection: {len(raw_records['ids'])}")
print("First three ids from get():")
for doc_id in raw_records['ids'][:3]:
    print(doc_id)

Total records in collection: 10
First three ids from get():
68a3c254-572e-4b13-841e-5a3b13d010ae
85c1ed09-19b7-47a0-a756-71d9d44d56de
fd837776-5bae-47f9-8ffc-7d951da17fe9


In [24]:
## pick few ids so we can read them back in a higher-level format
selected_ids = document_ids[:3]
selected_ids

['68a3c254-572e-4b13-841e-5a3b13d010ae',
 '85c1ed09-19b7-47a0-a756-71d9d44d56de',
 'fd837776-5bae-47f9-8ffc-7d951da17fe9']

In [25]:
## get by ids
selected_docs = vector_store.get_by_ids(selected_ids)
print_documents("Documents fetched with get_by_ids():", selected_docs)

Documents fetched with get_by_ids():
1. id=68a3c254-572e-4b13-841e-5a3b13d010ae
  topic=AI | doc_number=1
  content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
2. id=85c1ed09-19b7-47a0-a756-71d9d44d56de
  topic=AI | doc_number=2
  content=AI systems can analyze patterns in data to support predictions and automation.
3. id=fd837776-5bae-47f9-8ffc-7d951da17fe9
  topic=AI | doc_number=3
  content=Responsible AI development includes fairness, transparency, and safety checks.



## Run a Similarity Search

In [26]:
query = "How does RAG help an LLM answer questions usingn outside knowledge?"
query

'How does RAG help an LLM answer questions usingn outside knowledge?'

In [29]:
search_result = vector_store.similarity_search(query=query, k=3)
print(f"Query: {query}")
print_documents("Search results:", search_result)

Query: How does RAG help an LLM answer questions usingn outside knowledge?
Search results:
1. id=20031b4b-6824-4527-8e99-66b084cd48fc
  topic=RAG | doc_number=4
  content=RAG combines retrieval with generation so the model can answer using external knowledge.
2. id=9722a2a8-ca98-454e-b504-2d02b4ed3952
  topic=LLM | doc_number=8
  content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
3. id=1e67aaf7-2a83-4c7b-bb84-9ca3e47263a3
  topic=RAG | doc_number=5
  content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



In [30]:
search_result

[Document(id='20031b4b-6824-4527-8e99-66b084cd48fc', metadata={'topic': 'RAG', 'doc_number': 4}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
 Document(id='9722a2a8-ca98-454e-b504-2d02b4ed3952', metadata={'topic': 'LLM', 'doc_number': 8}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
 Document(id='1e67aaf7-2a83-4c7b-bb84-9ca3e47263a3', metadata={'doc_number': 5, 'topic': 'RAG'}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.')]

In [31]:
vector_store.similarity_search_with_score(query=query, k=3)

[(Document(id='20031b4b-6824-4527-8e99-66b084cd48fc', metadata={'topic': 'RAG', 'doc_number': 4}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
  0.8074676990509033),
 (Document(id='9722a2a8-ca98-454e-b504-2d02b4ed3952', metadata={'topic': 'LLM', 'doc_number': 8}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
  0.9337143898010254),
 (Document(id='1e67aaf7-2a83-4c7b-bb84-9ca3e47263a3', metadata={'topic': 'RAG', 'doc_number': 5}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'),
  1.0853300094604492)]

## Update existing documents

In [32]:
# we will update one RAG document and one LLM document
ids_to_update = [document_ids[3], document_ids[7]]
ids_to_update

['20031b4b-6824-4527-8e99-66b084cd48fc',
 '9722a2a8-ca98-454e-b504-2d02b4ed3952']

In [35]:
## keep the replacement text separate so the update step stays easy to follow
updated_examples = [
    {
        "id": ids_to_update[0],
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG improves answer quality by retrieving relevant context before the language model generates a response.",
    },
    {
        "id": ids_to_update[1],
        "topic": "LLM",
        "doc_number": 8,
        "text": "Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.",
    },
]

updated_docs = [
    Document(
        id=item['id'],
        page_content=item['text'],
        metadata={'topic': item['topic'], 'doc_number': item['doc_number']},
    )
    for item in updated_examples
]
print_documents("Updatedd document content:", updated_docs)

Updatedd document content:
1. id=20031b4b-6824-4527-8e99-66b084cd48fc
  topic=RAG | doc_number=4
  content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=9722a2a8-ca98-454e-b504-2d02b4ed3952
  topic=LLM | doc_number=8
  content=Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.



In [36]:
vector_store.update_documents(ids=ids_to_update, documents=updated_docs)

print("Updated these ids: ")
for doc_id in ids_to_update:
    print(doc_id)

Updated these ids: 
20031b4b-6824-4527-8e99-66b084cd48fc
9722a2a8-ca98-454e-b504-2d02b4ed3952


In [39]:
## read the updated records

updated_raw_records = vector_store.get(ids=ids_to_update)

print("Raw records returned by get(ids=ids_to__update):")
for doc_id, document_text, metadata in zip(
    updated_raw_records['ids'],
    updated_raw_records['documents'],
    updated_raw_records['metadatas'],
):
    print(f"id={doc_id}")
    print(f"metadata={metadata}")
    print(f"content={preview_text(document_text)}")
    print()

Raw records returned by get(ids=ids_to__update):
id=20031b4b-6824-4527-8e99-66b084cd48fc
metadata={'doc_number': 4, 'topic': 'RAG'}
content=RAG improves answer quality by retrieving relevant context before the language m...

id=9722a2a8-ca98-454e-b504-2d02b4ed3952
metadata={'doc_number': 8, 'topic': 'LLM'}
content=Well-written prompts help an LLM stay focused, follow instructions, and produce ...



### Delete Documents

In [40]:
ids_to_delete = [document_ids[8], document_ids[9]]
ids_to_delete

['1713080c-7c8b-4f40-bd4c-49b24fab3ed9',
 'cebc3992-fb84-4714-83d1-23e0dbf11189']

In [41]:
vector_store.delete(ids=ids_to_delete)

print("Deleted these ids:")
for doc_id in ids_to_delete:
    print(doc_id)

Deleted these ids:
1713080c-7c8b-4f40-bd4c-49b24fab3ed9
cebc3992-fb84-4714-83d1-23e0dbf11189


In [42]:
remaining_records = vector_store.get()
remaining_ids = [remaining_records['ids']]

print(f"Remaining document count: {len(remaining_ids)}")
print("Remaining ids:")
for doc_id in remaining_ids:
    print(doc_id)
    
print("\nDeleted ids still present?")
for doc_id in ids_to_delete:
    print(f"{doc_id}: {doc_id in remaining_ids}")

Remaining document count: 1
Remaining ids:
['68a3c254-572e-4b13-841e-5a3b13d010ae', '85c1ed09-19b7-47a0-a756-71d9d44d56de', 'fd837776-5bae-47f9-8ffc-7d951da17fe9', '20031b4b-6824-4527-8e99-66b084cd48fc', '1e67aaf7-2a83-4c7b-bb84-9ca3e47263a3', '435e964b-65c4-475d-88fb-f0de1c87c08a', 'fb541605-0f78-4f91-8418-25027d9f081f', '9722a2a8-ca98-454e-b504-2d02b4ed3952']

Deleted ids still present?
1713080c-7c8b-4f40-bd4c-49b24fab3ed9: False
cebc3992-fb84-4714-83d1-23e0dbf11189: False
